# PICKO Research · NB3 — **Semantic separation, k-fold CV**: a robust estimate on little data

## 0 · Colab quick-start (GPU) — run & forget, restart-safe

**On Colab first: Runtime → Change runtime type → GPU (L4 recommended; T4/A100 also fine).**
This cell clones **upstream Needle** (Cactus, pinned commit) and installs it, clones this **PICKO** repo,
pins the exact JAX/Flax, mounts Drive, and points **both** the data (in) and the checkpoints+results (out)
at your **`MyDrive/picko/`** folder — so a runtime restart loses nothing.

**Prerequisite (one-time):** `picko_training_pool.jsonl` must be in `MyDrive/picko/`. **Running locally?**
This cell is a no-op — install Needle yourself (`pip install -e /path/to/needle`) and skip to cell 1.

In [ ]:
# --- Colab bootstrap (safe to re-run; no-op locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
# Upstream Needle (Cactus) — de-vendored: cloned + installed at a pinned commit.
NEEDLE_REPO = "https://github.com/cactus-compute/needle.git"
NEEDLE_SHA  = "34861f39ae292429f80a62c96abe83218a852d57"   # pinned; has _per_tool_split — update if upstream drifts
# This PICKO repo — the scripts/notebooks/data imported below.
PICKO_REPO   = "https://github.com/hodayastern/Picko.git"  # this submission repo
PICKO_BRANCH = "main"
if IN_COLAB:
    if not os.path.exists("/content/needle"):
        !git clone -q {NEEDLE_REPO} /content/needle && cd /content/needle && git checkout -q {NEEDLE_SHA}
    if not os.path.exists("/content/picko"):
        !git clone -q -b {PICKO_BRANCH} {PICKO_REPO} /content/picko
    %pip install -q "jax[cuda12]==0.10.2" "jaxlib==0.10.2" "flax==0.12.8"
    %pip install -q -e /content/needle                          # install upstream Needle
    sys.path.insert(0, "/content/picko")
    from google.colab import drive; drive.mount("/content/drive")
    import shutil
    DRIVE = "/content/drive/MyDrive/picko"                      # <- everything lives here
    os.environ["PICKO_OUT_DIR"] = f"{DRIVE}/picko_out"          # checkpoints + results (durable)
    os.environ["PICKO_LOG"]     = f"{DRIVE}/picko_out/run.log"  # durable log across restarts
    os.makedirs(os.environ["PICKO_OUT_DIR"], exist_ok=True)
    dst = "/content/picko/data/picko_training_pool.jsonl"
    if not os.path.exists(dst):
        cands = [f"{DRIVE}/picko_training_pool.jsonl", "/content/drive/MyDrive/picko_training_pool.jsonl"]
        src = next((c for c in cands if os.path.exists(c)), None)
        if src is None:
            have = os.listdir(DRIVE) if os.path.isdir(DRIVE) else "(MyDrive/picko not found)"
            raise FileNotFoundError(
                "picko_training_pool.jsonl not found. Upload it to MyDrive/picko/. "
                f"Currently in {DRIVE}: {have}")
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst)
        print("copied data from", src)
    try:                                                        # guard: upstream must expose the split PICKO uses
        from needle.training.finetune import _per_tool_split    # noqa: F401
    except Exception as e:
        raise ImportError(f"Upstream Needle @ {NEEDLE_SHA[:7]} lacks _per_tool_split ({e}). "
                          "Pin NEEDLE_SHA to a commit that has it, or re-run the install.")
    import jax
    print("GPU:");
    !nvidia-smi -L
    print("jax devices:", jax.devices())
    _plat = jax.devices()[0].platform
    assert _plat == "gpu", (
        f"JAX is running on '{_plat}', NOT the GPU — every finetune/eval will be ~30x slower "
        "(hours instead of minutes). FIX: Runtime > Change runtime type > GPU (L4), then "
        "Runtime > Restart session, and re-run this cell. If a GPU IS selected but this still "
        "fails, the CUDA plugin didn't load — re-run the %pip lines above, then restart.")
    print(f"bootstrap OK · GPU active · needle@{NEEDLE_SHA[:7]} · data =", dst, "· OUT_DIR =", os.environ["PICKO_OUT_DIR"])
else:
    print("Not on Colab — running locally. Install upstream Needle first: pip install -e /path/to/needle")

## 1 · Setup & data overview

In [ ]:
# ensure the repo root is importable (works from notebooks/research/, Colab, etc.)
import os, sys
_here = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.exists(os.path.join(_here, "scripts", "picko_research.py")): break
    _here = os.path.dirname(_here)
if os.path.isdir("/content/picko"): _here = "/content/picko"
if _here not in sys.path: sys.path.insert(0, _here)

from scripts.picko_research import *
import json, time
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None
from tqdm.auto import tqdm

cat, tok, raw, FOCUS, OUT_DIR = load_context()
env_report(OUT_DIR)   # jax devices + is OUT_DIR durable (Drive)?

## 2 · Why k-fold here

Can the model *learn* to route by domain meaning alone (no source name in the query)? With only a couple
hundred source-free queries, a single test split gives one shaky number. **We run the experiment as k-fold
cross-validation:** every query serves as test **exactly once**, and we report the mean over folds with a
std error bar — a distribution instead of a single point, which is what small data needs.

Per fold we train a fresh 2-tool model on the training folds and evaluate it on the held-out fold.
`before` = the existing name-trained model on the same held-out fold (no semantic training); `after` = the
fold's freshly trained model. We aggregate all folds at the end.

## 3 · Build the source-free dataset

Reads the two domain query files (`data/semantic/cs.json` → arxiv, `data/semantic/medicine.json` → pubmed),
drops any query that still names a source (safety net), and bakes the offered pair (full schema, order
randomised) into each row — the standard `{query, tools, answers}` format. Only the **unnamed** (source-free)
queries are used here. Falls back to the prebuilt `data/semantic/probe.jsonl` if the raw files aren't present.

In [ ]:
import re, collections, random as _rnd
BANNED = r"\b(arxiv|arxiv\.org|pubmed|medline|ncbi|pmid|biorxiv|medrxiv|preprint|wikipedia|wiki|hugging\s*face|huggingface|semantic scholar)\b"
DOMAIN = {"arxiv_search_papers": ("cs", "arXiv"), "pubmed_search_articles": ("medicine", "PubMed")}
RAWFILE = {"arxiv_search_papers": "semantic/cs.json", "pubmed_search_articles": "semantic/medicine.json"}
PAIR_TOOLS = list(DOMAIN)                       # the forced 2-way choice

def _find(name):
    for root in [os.path.join(ROOT, "data"), "/content/drive/MyDrive/picko",
                 "/content/drive/MyDrive/picko/data"]:
        p = os.path.join(root, name)
        if os.path.exists(p): return p
    return None

def _tools_json(seed):                          # offered pair (full schema), order randomised per row
    ts = [cat.by_name[n] for n in PAIR_TOOLS]
    _rnd.Random(seed).shuffle(ts)
    return json.dumps(ts, separators=(",", ":"), ensure_ascii=False)

raw_paths = {g: _find(f) for g, f in RAWFILE.items()}
if all(raw_paths.values()):
    DATA, seen, i = [], set(), 0
    for gold, (domain, src) in DOMAIN.items():
        ans = json.dumps([{"name": gold, "arguments": {}}])
        for q in json.load(open(raw_paths[gold])):
            q = q.strip(); key = re.sub(r"\W+", " ", q.lower()).strip()
            if not q or key in seen or re.search(BANNED, q, re.I): continue
            seen.add(key)
            DATA.append({"query": q, "tools": _tools_json(i), "answers": ans,
                         "gold": gold, "domain": domain}); i += 1
    pp = os.path.join(ROOT, "data", "semantic", "train.jsonl")
    os.makedirs(os.path.dirname(pp), exist_ok=True)
    with open(pp, "w") as f:
        for r in DATA: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    log(f"built source-free dataset ({len(DATA)} rows) → {pp}")
else:
    pp = _find("semantic/probe.jsonl")
    if not pp: raise FileNotFoundError("need data/semantic/cs.json + medicine.json (or probe.jsonl) in data/ or Drive")
    DATA = [r for r in (json.loads(l) for l in open(pp) if l.strip()) if r.get("condition", "unnamed") == "unnamed"]
    log(f"loaded {len(DATA)} source-free rows from {pp}")

df = pd.DataFrame(DATA)
print("dataset:", len(DATA), "| per domain:", df["domain"].value_counts().to_dict())
display(df[["query", "gold", "domain"]].head(6))

## 4 · Configure

In [ ]:
NB_DIR = os.path.join(OUT_DIR, "nb3"); os.makedirs(NB_DIR, exist_ok=True)   # this notebook's outputs
K             = 5      # number of folds (each query is test exactly once)
EPOCHS        = 3
BATCH_SIZE    = 8
MAX_GEN_LEN   = 64     # we only score tool SELECTION
SPLIT_SEED    = 42
RUN_TRAIN     = True
FORCE_RETRAIN = False
BASELINE_CKPT = next((c for c in [os.path.join(OUT_DIR, "nb2", "picko_depth_focus40_best.pkl"),
                                  os.path.join(OUT_DIR, "nb3_separation", "picko_focus40_best.pkl"),
                                  os.path.join(OUT_DIR, "nb1", "picko_breadth_focus40_best.pkl")]
                      if os.path.exists(c)), None)
print("pair:", PAIR_TOOLS, "| K:", K, "| epochs:", EPOCHS, "| baseline:", BASELINE_CKPT, "| out:", NB_DIR)

## 5 · Stratified k-fold split (per tool, non-overlapping)

Each tool's queries are shuffled deterministically and dealt round-robin into K folds, so every fold holds
~1/K of **each** tool (balanced), the folds never overlap, and together they cover the whole dataset.

In [ ]:
import collections, random as _rk
by_tool = collections.defaultdict(list)
for idx, e in enumerate(DATA): by_tool[e["gold"]].append(idx)
FOLDS = [[] for _ in range(K)]
for tool, idxs in by_tool.items():
    idxs = list(idxs); _rk.Random(SPLIT_SEED).shuffle(idxs)
    for j, ix in enumerate(idxs): FOLDS[j % K].append(ix)      # round-robin -> balanced, non-overlapping
allix = [j for f in FOLDS for j in f]
assert len(allix) == len(set(allix)) == len(DATA), "folds overlap or miss examples"
print("each of", len(DATA), "examples is in exactly one fold")
for i, f in enumerate(FOLDS):
    print(f"  fold {i}: n={len(f)} · per domain", dict(collections.Counter(DATA[j]["domain"] for j in f)))

## 6 · Run the folds — train and evaluate on the held-out fold

For each fold: build train (the other folds) + test (this fold); train a fresh 2-tool model on train;
predict on the held-out test; also run the `before` baseline on it. Resumable — a finished fold is loaded
from `nb3/kfold_results.json` and its training is skipped.

In [ ]:
import contextlib, io
RES = os.path.join(NB_DIR, "kfold_results.json")
prev = json.load(open(RES)) if (os.path.exists(RES) and not FORCE_RETRAIN) else {"folds": []}
fold_rows = {r["fold"]: r for r in prev["folds"]}
if fold_rows: log(f"resumed {len(fold_rows)} finished fold(s)")

bm = None
if BASELINE_CKPT: bm, bp, btk = load_model(BASELINE_CKPT)
_acc = lambda sel: (float(np.mean(sel)) if len(sel) else float("nan"))

t_all = time.time()
for i in range(K):
    if i in fold_rows and not FORCE_RETRAIN:
        log(f"fold {i}: skip (done)"); continue
    try:
        test_ix = set(FOLDS[i])
        train = [DATA[j] for j in range(len(DATA)) if j not in test_ix]
        test  = [DATA[j] for j in FOLDS[i]]
        trq = {e["query"] for e in train}                         # held-out queries must be disjoint from train
        assert not [e for e in test if e["query"] in trq], f"fold {i}: train/test overlap"
        log(f"fold {i}: train={len(train)} test={len(test)}")
        # --- AFTER: fresh 2-tool model trained on the training folds (checkpoint reused via finetune_and_eval) ---
        R = finetune_and_eval(cat, raw, tok, PAIR_TOOLS, f"kfold{i}", NB_DIR,
                              dataset=train, epochs=EPOCHS, run_train=RUN_TRAIN,
                              force_retrain=FORCE_RETRAIN, eval_subsample=1,
                              max_gen_len=MAX_GEN_LEN, batch_size=BATCH_SIZE)
        am, ap, atk = R["bundle"]
        with contextlib.redirect_stdout(io.StringIO()):
            after_preds = predict(am, ap, atk, test, max_gen_len=MAX_GEN_LEN, batch=BATCH_SIZE)
        after_sel = [s["selected"] for s in evaluate_per_example(test, after_preds)]
        # --- BEFORE: existing name-trained model on the same held-out fold (no training) ---
        if bm is not None:
            with contextlib.redirect_stdout(io.StringIO()):
                before_preds = predict(bm, bp, btk, test, max_gen_len=MAX_GEN_LEN, batch=BATCH_SIZE)
            before_sel = [s["selected"] for s in evaluate_per_example(test, before_preds)]
        else:
            before_sel = None
        doms = [e["domain"] for e in test]
        row = {"fold": i, "n": len(test), "after_overall": _acc(after_sel),
               "before_overall": (_acc(before_sel) if before_sel is not None else None),
               "confusion": confusion(test, after_preds)}
        for d in ["cs", "medicine"]:
            row[f"after_{d}"]  = _acc([after_sel[k] for k, dm in enumerate(doms) if dm == d])
            row[f"before_{d}"] = (_acc([before_sel[k] for k, dm in enumerate(doms) if dm == d])
                                  if before_sel is not None else None)
        fold_rows[i] = row
        json.dump({"folds": [fold_rows[k] for k in sorted(fold_rows)]}, open(RES, "w"), indent=2)
        log(f"fold {i}: after_overall={row['after_overall']:.3f}"
            + (f" before={row['before_overall']:.3f}" if before_sel is not None else ""))
    except Exception as ex:
        log(f"fold {i}: FAILED ({type(ex).__name__}: {ex})")
log(f"ALL {K} FOLDS DONE in {time.time()-t_all:.0f}s · results={RES}")

## 7 · Aggregate across folds (weight the iterations)

`overall`/`cs`/`medicine` are averaged over the K folds -> **mean ± std** (the std is the across-fold
uncertainty). The confusion matrix pools every fold's held-out predictions, so each query is counted once
(a true out-of-fold routing matrix over the whole dataset).

In [ ]:
rows = [fold_rows[k] for k in sorted(fold_rows)]
fr = pd.DataFrame(rows)
have_before = fr["before_overall"].notna().any()
def _ms(col):
    v = fr[col].dropna().values
    return (round(float(np.mean(v)), 4), round(float(np.std(v)), 4)) if len(v) else (float("nan"), float("nan"))
cats = ["overall", "cs", "medicine"]
summary = []
for phase in (["before", "after"] if have_before else ["after"]):
    r = {"phase": phase}
    for c in cats:
        mu, sd = _ms(f"{phase}_{c}"); r[c] = mu; r[c + "_std"] = sd
    summary.append(r)
POOLED = {}
for row in rows:
    for t, d in row.get("confusion", {}).items():
        POOLED.setdefault(t, {})
        for p, c in d.items(): POOLED[t][p] = POOLED[t].get(p, 0) + c
json.dump({"pair": PAIR_TOOLS, "K": K, "folds": rows, "summary": summary, "confusion_oof": POOLED},
          open(RES, "w"), indent=2)
log(f"aggregated {len(rows)} folds -> {RES}")
display(pd.DataFrame(summary))

## 8 · Plot — before vs after (mean ± std over folds) + pooled out-of-fold routing

In [ ]:
x = np.arange(len(cats)); w = 0.8 / max(len(summary), 1)
palette = {"before": "#8a8a8a", "after": "#009E73"}
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1.5, 1]})
for j, r in enumerate(summary):
    vals = [r.get(c, np.nan) for c in cats]; errs = [r.get(c + "_std", 0) for c in cats]
    off = (j - (len(summary) - 1) / 2) * w
    ax.bar(x + off, vals, w, yerr=errs, capsize=4, color=palette.get(r["phase"], "#0072B2"),
           edgecolor="white", lw=0.6, label=r["phase"])
    for k, v in enumerate(vals):
        if not np.isnan(v): ax.text(x[k] + off, min(v + errs[k] + 0.03, 1.05), f"{v:.2f}", ha="center", fontsize=9)
ax.axhline(0.5, color="#C44E52", ls="--", lw=1.2, label="chance (2-way)")
ax.set_xticks(x); ax.set_xticklabels(cats); ax.set_ylim(0, 1.12); ax.set_ylabel("Tool-selection accuracy")
ax.set_title(f"Semantic routing — {K}-fold CV (mean +/- std)"); ax.legend(loc="lower right")
if sns: sns.despine(ax=ax)
ax.grid(axis="y", color="#cccccc", lw=0.6, alpha=0.6); ax.set_axisbelow(True)

M = pd.DataFrame(0, index=PAIR_TOOLS, columns=PAIR_TOOLS)
for t, d in POOLED.items():
    for p, c in d.items():
        if t in PAIR_TOOLS and p in PAIR_TOOLS: M.loc[t, p] = c
short = lambda n: n.replace("_search_papers", "").replace("_search_articles", "")
if sns:
    sns.heatmap(M.div(M.sum(1).replace(0, 1), axis=0), cmap="Greens", vmin=0, vmax=1, cbar=False,
                annot=M.values, fmt="d", linewidths=0.5, linecolor="white", ax=ax2,
                xticklabels=[short(c) for c in M.columns], yticklabels=[short(r) for r in M.index])
ax2.set_title("Out-of-fold routing (all folds pooled)"); ax2.set_xlabel("predicted"); ax2.set_ylabel("true domain tool")
plt.tight_layout(); save_fig("semantic_kfold", out_dir=NB_DIR); plt.show()

## 9 · Read-out

Every source-free query is tested exactly once by a model that never trained on it, so the k-fold mean is a
trustworthy estimate on this small dataset. If **after** sits clearly above **before** and above the 0.5
chance line across folds — with a small std — the domain distinction is genuinely learnable; a large std or
overlap with chance means the signal is weak for a 26M model on this little data.